XGBoost Model

In [1]:
#pip install xgboost optuna scikit-learn
import pandas as pd
import numpy as np
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import average_precision_score, roc_auc_score, confusion_matrix, classification_report, precision_recall_curve

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 40

Loading the Dataframe

In [2]:
#this is the dataframe made at the end of sample_new_labels.ipynb
df = pd.read_csv("model_dataframe_2023_Q1_to_2026_Q3.txt", sep="\t", low_memory=False, parse_dates=["filed_date"])

#I keep the rows where we know the answer (filed at least a year ago) and the company wasn't already bankrupt
df = df[df["label_known"] & ~df["already_bankrupt"]].reset_index(drop=True)
print(df.shape)
print(df["Bankruptcy"].value_counts())
print("companies:", df["cik"].nunique(), " companies with a 1:", df.loc[df["Bankruptcy"] == 1, "cik"].nunique())

(64537, 23)
Bankruptcy
0.0    63928
1.0      609
Name: count, dtype: int64
companies: 7137  companies with a 1: 204


Features

In [3]:
#only the 8 variables we agreed on, nothing else
features = ["total_assets", "current_assets", "current_liabilities", "total_liabilities",
            "revenue", "net_income", "operating_income", "operating_cash_flow"]
df[features].describe().T

,count,mean,std,min,25%,50%,75%,max
total_assets,63982.0,1.306428e+10,1.169734e+11,0.000000e+00,4.239137e+07,5.366950e+08,3.586198e+09,4.552482e+12
current_assets,50473.0,1.768473e+09,8.406705e+09,-4.900000e+01,7.427105e+06,1.399440e+08,7.843000e+08,3.438409e+11
current_liabilities,50297.0,1.421983e+09,7.512260e+09,0.000000e+00,5.812574e+06,4.847000e+07,4.236973e+08,3.357346e+11
total_liabilities,63949.0,1.041419e+10,1.086209e+11,-4.943100e+07,1.241072e+07,2.244850e+08,2.181378e+09,4.255655e+12
revenue,47661.0,5.090657e+09,2.482924e+10,-2.996000e+09,1.711333e+07,3.531080e+08,2.270336e+09,6.794620e+11
net_income,63526.0,3.684305e+08,3.290572e+09,-4.764400e+10,-1.868764e+07,-2.061567e+04,8.389900e+07,1.453200e+11
operating_income,49231.0,4.215996e+08,3.747742e+09,-2.641800e+10,-2.216700e+07,-9.387560e+05,9.072050e+07,1.713280e+11
operating_cash_flow,63373.0,4.427456e+08,7.053461e+09,-1.007356e+12,-9.393648e+06,5.148040e+05,1.571570e+08,1.595800e+11


Train and Test Split

In [4]:
#one company can have up to 4 rows with a 1 for the same bankruptcy, and its numbers look almost the same in each.
#If some of them are in train and some in test, the model has already seen that company, so the test score is too
#good. So I split by company: every row of a company is either in train or in test (20% of companies in test)
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_index, test_index = next(splitter.split(df, df["Bankruptcy"], groups=df["cik"]))
train = df.iloc[train_index].reset_index(drop=True)
test = df.iloc[test_index].reset_index(drop=True)

print("train:", len(train), "rows,", int(train["Bankruptcy"].sum()), "ones,", train["cik"].nunique(), "companies")
print("test:", len(test), "rows,", int(test["Bankruptcy"].sum()), "ones,", test["cik"].nunique(), "companies")
print("companies in both:", len(set(train["cik"]) & set(test["cik"])))

train: 51633 rows, 490 ones, 5705 companies
test: 12904 rows, 119 ones, 1432 companies
companies in both: 0


Ways to Handle the Imbalance

In [5]:
#only about 1% of rows are 1, so I try 4 ways:
#  none        - train as it is
#  weights     - give the 1s more weight (scale_pos_weight), the model pays more for missing a 1
#  oversample  - copy the 1s (random rows with repeats) until there are more of them
#  undersample - keep all the 1s and only a random part of the 0s
#The copying and dropping is only done on the training part, never on the rows we score.
#I didn't use SMOTE because it can't work with empty values, and filling them would hide which companies
#don't report a number (that is useful information for the model)
def balance_rows(X, y, method, ratio, seed=42):
    rng = np.random.default_rng(seed)
    ones = np.where(y == 1)[0]
    zeros = np.where(y == 0)[0]
    if method == "oversample":
        #ratio = how many 1s we want for each 0
        extra = rng.choice(ones, size=max(int(ratio * len(zeros)) - len(ones), 0), replace=True)
        keep = np.concatenate([zeros, ones, extra])
    elif method == "undersample":
        #ratio = how many 1s for each 0 that we keep
        keep = np.concatenate([rng.choice(zeros, size=min(int(len(ones) / ratio), len(zeros)), replace=False), ones])
    else:
        keep = np.arange(len(y))
    keep = rng.permutation(keep)
    return X.iloc[keep], y.iloc[keep]

Optuna Search

In [6]:
def make_params(trial, method):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 2, 8),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 30, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        #the xgboost docs say max_delta_step helps logistic regression when the classes are very imbalanced
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
    }
    ratio = None
    if method == "weights":
        #0s / 1s is about 105, so I let it go from 1 (no weight) to a bit above that
        params["scale_pos_weight"] = trial.suggest_float("scale_pos_weight", 1, 150, log=True)
    if method == "oversample":
        ratio = trial.suggest_float("ratio", 0.05, 1.0, log=True)
    if method == "undersample":
        ratio = trial.suggest_float("ratio", 0.02, 1.0, log=True)
    return params, ratio


def new_model(params):
    return XGBClassifier(**params, tree_method="hist", eval_metric="aucpr", n_jobs=8, random_state=42)


def cv_scores(params, ratio, method, data, features, folds=4):
    #4 fold cross validation inside the training companies, again split by company.
    #Returns the PR-AUC of each fold and the score of every row when it was in the validation fold
    cv = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=7)
    scores = np.zeros(len(data))
    fold_pr_auc = []
    for fit_index, val_index in cv.split(data, data["Bankruptcy"], groups=data["cik"]):
        X_fit, y_fit = balance_rows(data.loc[fit_index, features], data.loc[fit_index, "Bankruptcy"], method, ratio)
        model = new_model(params).fit(X_fit, y_fit)
        scores[val_index] = model.predict_proba(data.loc[val_index, features])[:, 1]
        fold_pr_auc.append(average_precision_score(data.loc[val_index, "Bankruptcy"], scores[val_index]))
        yield fold_pr_auc, scores


def run_study(method, features, n_trials=N_TRIALS):
    def objective(trial):
        params, ratio = make_params(trial, method)
        for step, (fold_pr_auc, _) in enumerate(cv_scores(params, ratio, method, train, features)):
            #stop bad trials early
            trial.report(np.mean(fold_pr_auc), step)
            if trial.should_prune():
                raise optuna.TrialPruned()
        return np.mean(fold_pr_auc)

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1))
    study.optimize(objective, n_trials=n_trials)
    return study

In [7]:
#one Optuna search for each way of handling the imbalance, all with the same features (this takes a while)
studies = {}
for method in ["none", "weights", "oversample", "undersample"]:
    studies[method] = run_study(method, features)
    print(method, "best cross validation PR-AUC:", round(studies[method].best_value, 4))

none best cross validation PR-AUC: 0.0798


weights best cross validation PR-AUC: 0.0749


oversample best cross validation PR-AUC: 0.0759


undersample best cross validation PR-AUC: 0.0796


Scores on the Test Companies

In [8]:
def top_share_recall(y, scores, share):
    #if the investigator agent only looks at the riskiest x% of filings, how many of the 1s does it find
    cutoff = np.quantile(scores, 1 - share)
    return y[scores >= cutoff].sum() / y.sum()


def evaluate(method, study, features, train_data, test_data):
    params = {k: v for k, v in study.best_params.items() if k != "ratio"}
    ratio = study.best_params.get("ratio")
    #threshold: the one with the best F1 on the cross validation scores of the training companies (not the test ones)
    *_, (_, cv_score) = cv_scores(params, ratio, method, train_data, features)
    precision, recall, thresholds = precision_recall_curve(train_data["Bankruptcy"], cv_score)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    threshold = thresholds[np.argmax(f1[:-1])]

    X_fit, y_fit = balance_rows(train_data[features], train_data["Bankruptcy"], method, ratio)
    model = new_model(params).fit(X_fit, y_fit)
    scores = model.predict_proba(test_data[features])[:, 1]
    y = test_data["Bankruptcy"].values
    predicted = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, predicted).ravel()

    #a bankrupt company counts as found if at least one of its filings in the year before is above the threshold
    companies = test_data.assign(score=scores)[y == 1].groupby("cik")["score"].max()
    row = {"method": method, "cv PR-AUC": study.best_value,
           "test PR-AUC": average_precision_score(y, scores), "test ROC-AUC": roc_auc_score(y, scores),
           "precision": tp / max(tp + fp, 1), "recall": tp / (tp + fn), "F1": 2 * tp / (2 * tp + fp + fn),
           "companies found": f"{(companies >= threshold).sum()} of {len(companies)}",
           "recall top 1%": top_share_recall(y, scores, 0.01), "recall top 5%": top_share_recall(y, scores, 0.05),
           "recall top 10%": top_share_recall(y, scores, 0.10)}
    return row, model, scores, threshold


results = {}
for method, study in studies.items():
    results[method] = evaluate(method, study, features, train, test)

#PR-AUC by chance is just the share of 1s, which is about 0.01 here
print("share of 1s in test:", round(test["Bankruptcy"].mean(), 4))
comparison = pd.DataFrame([r[0] for r in results.values()]).set_index("method").round(3)
comparison

share of 1s in test: 0.0092


,cv PR-AUC,test PR-AUC,test ROC-AUC,precision,recall,F1,companies found,recall top 1%,recall top 5%,recall top 10%
method,,,,,,,,,,
none,0.080,0.119,0.884,0.099,0.210,0.135,13 of 40,0.151,0.471,0.714
weights,0.075,0.119,0.894,0.100,0.286,0.148,17 of 40,0.160,0.496,0.731
oversample,0.076,0.114,0.887,0.092,0.277,0.138,17 of 40,0.134,0.479,0.706
undersample,0.080,0.088,0.889,0.100,0.328,0.154,20 of 40,0.160,0.496,0.689


In [9]:
#best settings for each method
pd.DataFrame({m: s.best_params for m, s in studies.items()}).round(4)

,none,weights,oversample,undersample
n_estimators,100.0000,900.0000,950.0000,300.0000
learning_rate,0.1879,0.0498,0.0135,0.0853
max_depth,2.0000,2.0000,3.0000,3.0000
min_child_weight,4.8327,11.3123,1.1663,27.3184
subsample,0.5917,0.8804,0.6627,0.5397
colsample_bytree,0.6321,0.7368,0.6332,0.4853
gamma,3.7082,3.8548,1.3567,3.3770
reg_alpha,0.0295,0.0944,2.0651,0.0245
reg_lambda,30.5890,0.8582,0.2087,0.3376
max_delta_step,2.0000,4.0000,3.0000,2.0000


Stable Score (10 rounds)

In [10]:
#the test set above has only about 40 bankrupt companies, so one split can be lucky or unlucky.
#Here I split by company 5 ways, twice with different shuffles (10 rounds), using weights and its best settings,
#and take the average
best_method = "weights"
params = {k: v for k, v in studies[best_method].best_params.items() if k != "ratio"}
ratio = studies[best_method].best_params.get("ratio")
rounds = []
for seed in [1, 2]:
    for fit_index, val_index in StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed).split(df, df["Bankruptcy"], groups=df["cik"]):
        X_fit, y_fit = balance_rows(df.loc[fit_index, features], df.loc[fit_index, "Bankruptcy"], best_method, ratio)
        round_scores = new_model(params).fit(X_fit, y_fit).predict_proba(df.loc[val_index, features])[:, 1]
        y = df.loc[val_index, "Bankruptcy"].values
        rounds.append({"PR-AUC": average_precision_score(y, round_scores), "ROC-AUC": roc_auc_score(y, round_scores),
                       "recall top 5%": top_share_recall(y, round_scores, 0.05),
                       "recall top 10%": top_share_recall(y, round_scores, 0.10)})
rounds = pd.DataFrame(rounds)
pd.DataFrame({"average": rounds.mean(), "spread (std)": rounds.std()}).round(3)

,average,spread (std)
PR-AUC,0.099,0.025
ROC-AUC,0.894,0.015
recall top 5%,0.460,0.058
recall top 10%,0.640,0.070


Results of the Best Model

In [11]:
#all 4 ways score about the same, so I use weights since it's the simplest and doesn't throw away any rows
row, model, scores, threshold = results[best_method]
y_test = test["Bankruptcy"].values
predicted = (scores >= threshold).astype(int)
print("method:", best_method, " threshold:", round(threshold, 3))
print(confusion_matrix(y_test, predicted))
print(classification_report(y_test, predicted, digits=3))

method: weights  threshold: 0.082
[[12478   307]
 [   85    34]]
              precision    recall  f1-score   support

         0.0      0.993     0.976     0.985     12785
         1.0      0.100     0.286     0.148       119

    accuracy                          0.970     12904
   macro avg      0.546     0.631     0.566     12904
weighted avg      0.985     0.970     0.977     12904



In [12]:
#which features the model used the most (gain = how much each one improves the splits)
importance = pd.Series(model.get_booster().get_score(importance_type="gain")).reindex(features).fillna(0)
(importance / importance.sum()).sort_values(ascending=False).round(3).head(20)

net_income             0.283
operating_income       0.158
operating_cash_flow    0.127
current_liabilities    0.127
current_assets         0.094
total_liabilities      0.091
revenue                0.065
total_assets           0.052
dtype: float64

In [13]:
#the test filings with the highest scores
test.assign(score=scores).sort_values("score", ascending=False)[
    ["cik", "name", "form", "ddate", "filed", "score", "Bankruptcy", "days_to_bankruptcy"]].head(20)

,cik,name,form,ddate,filed,score,Bankruptcy,days_to_bankruptcy
997,105319,"WW INTERNATIONAL, INC.",10-Q,20240630,20240801,0.568470,1.0,279.0
999,105319,"WW INTERNATIONAL, INC.",10-K,20241231,20250228,0.510047,1.0,68.0
998,105319,"WW INTERNATIONAL, INC.",10-Q,20240930,20241106,0.502066,1.0,182.0
996,105319,"WW INTERNATIONAL, INC.",10-Q,20240331,20240502,0.440207,0.0,NaN
1000,105319,"WW INTERNATIONAL, INC.",10-Q,20250331,20250506,0.427142,1.0,1.0
11205,1838987,"COMPLETE SOLARIA, INC.",10-Q,20240630,20240814,0.372985,0.0,NaN
11898,1868640,ROADZEN INC.,10-Q,20240930,20241113,0.364457,0.0,NaN
12454,1929589,MARIADB PLC,10-Q,20240630,20240814,0.340034,0.0,NaN
10729,1820190,SCILEX HOLDING CO,10-Q,20250630,20250813,0.338808,0.0,NaN
12806,2013745,"CALUMET, INC. /DE",10-Q,20240930,20241112,0.336500,0.0,NaN


Check on Later Filings (split by date)

In [14]:
#a second check: train on filings from before July 2024 and test on the ones after, with the best settings.
#Some test 1s are the same bankruptcy as 1s in train (the company filed on both sides of the date),
#so I also show the score without those companies
early = df[df["filed_date"] < "2024-07-01"]
late = df[df["filed_date"] >= "2024-07-01"]
X_fit, y_fit = balance_rows(early[features], early["Bankruptcy"], best_method, ratio)
late_scores = new_model(params).fit(X_fit, y_fit).predict_proba(late[features])[:, 1]

seen = late["cik"].isin(early.loc[early["Bankruptcy"] == 1, "cik"])
for label, rows in [("all later filings", np.ones(len(late), bool)), ("without companies already 1 in train", ~seen.values)]:
    y = late["Bankruptcy"].values[rows]
    print(f"{label}: {len(y)} rows, {int(y.sum())} ones, PR-AUC {average_precision_score(y, late_scores[rows]):.3f}, "
          f"ROC-AUC {roc_auc_score(y, late_scores[rows]):.3f}, recall top 5% {top_share_recall(y, late_scores[rows], 0.05):.3f}")

all later filings: 28268 rows, 185 ones, PR-AUC 0.087, ROC-AUC 0.901, recall top 5% 0.492
without companies already 1 in train: 28208 rows, 128 ones, PR-AUC 0.060, ROC-AUC 0.885, recall top 5% 0.461


Scoring the New Filings

In [15]:
#the final model is trained on all the rows with a known answer, then I score the filings from the last year
#(we don't know their answer yet). These would go to the investigator agent
final_X, final_y = balance_rows(df[features], df["Bankruptcy"], best_method, ratio)
final_model = new_model(params).fit(final_X, final_y)

everything = pd.read_csv("model_dataframe_2023_Q1_to_2026_Q3.txt", sep="\t", low_memory=False)
new_filings = everything[~everything["label_known"] & ~everything["already_bankrupt"]].copy()
new_filings["score"] = final_model.predict_proba(new_filings[features])[:, 1]

#some of these companies already filed for bankruptcy after the filing (bk_date is filled), which is an early check
top = new_filings.sort_values("score", ascending=False)
print("new filings:", len(new_filings), " already bankrupt since then:", new_filings["bk_date"].notna().sum(),
      " of those in the top 100 scores:", top.head(100)["bk_date"].notna().sum())
top[["cik", "name", "form", "ddate", "filed", "score", "bk_date"]].head(20)

new filings: 22138  already bankrupt since then: 69  of those in the top 100 scores: 12


,cik,name,form,ddate,filed,score,bk_date
72602,1831481,SABLE OFFSHORE CORP.,10-Q,20260331,20260506,0.715674,NaN
61642,1705012,"FAT BRANDS, INC",10-Q,20250930,20251112,0.700912,2026-01-27
68697,1802450,"MSP RECOVERY, INC.",10-Q,20250930,20251119,0.662351,NaN
14096,827187,SLEEP NUMBER CORP,10-Q,20260331,20260512,0.623003,2026-06-12
5834,95552,SUPERIOR INDUSTRIES INTERNATIONAL INC,10-Q,20250930,20251219,0.606929,NaN
70603,1819516,WHEELS UP EXPERIENCE INC.,10-K,20251231,20260310,0.599068,NaN
70602,1819516,WHEELS UP EXPERIENCE INC.,10-Q,20250930,20251105,0.599068,NaN
70605,1819516,WHEELS UP EXPERIENCE INC.,10-Q,20260630,20260805,0.566289,NaN
66306,1769624,TRILLER GROUP INC.,10-K,20251231,20260414,0.561633,NaN
70604,1819516,WHEELS UP EXPERIENCE INC.,10-Q,20260331,20260511,0.560843,NaN


Nested Cross-Validation (the score we report)

In [16]:
#the scores above use settings that Optuna picked on the training companies, and some of those companies
#are test companies in the 10 rounds, so those scores can be a little too good. Here every round tunes on its own:
#  outer loop: 5 groups of companies, each one is the test group once
#  inner loop: Optuna only sees the other 4 groups (train on 3, validate on 1, rotated)
#so the test group is never used for training or tuning in its round, and every company is tested once
def nested_cv(method, n_trials=N_TRIALS):
    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=11)
    rows = []
    for fold, (fit_index, test_index) in enumerate(outer.split(df, df["Bankruptcy"], groups=df["cik"]), start=1):
        outer_train = df.iloc[fit_index].reset_index(drop=True)
        outer_test = df.iloc[test_index].reset_index(drop=True)

        def objective(trial):
            params, ratio = make_params(trial, method)
            for step, (fold_pr_auc, _) in enumerate(cv_scores(params, ratio, method, outer_train, features)):
                trial.report(np.mean(fold_pr_auc), step)
                if trial.should_prune():
                    raise optuna.TrialPruned()
            return np.mean(fold_pr_auc)

        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42),
                                    pruner=optuna.pruners.MedianPruner(n_startup_trials=8, n_warmup_steps=1))
        study.optimize(objective, n_trials=n_trials)
        params = {k: v for k, v in study.best_params.items() if k != "ratio"}
        ratio = study.best_params.get("ratio")
        X_fit, y_fit = balance_rows(outer_train[features], outer_train["Bankruptcy"], method, ratio)
        scores = new_model(params).fit(X_fit, y_fit).predict_proba(outer_test[features])[:, 1]
        y = outer_test["Bankruptcy"].values
        rows.append({"method": method, "fold": fold,
                     "ROC-AUC": roc_auc_score(y, scores), "PR-AUC": average_precision_score(y, scores),
                     "test companies": outer_test["cik"].nunique(),
                     "bankrupt companies in test": outer_test.loc[outer_test["Bankruptcy"] == 1, "cik"].nunique()})
        print(method, "fold", fold, "ROC-AUC", round(rows[-1]["ROC-AUC"], 3))
    return rows

nested_rows = []
for method in ["none", "weights", "oversample", "undersample"]:
    nested_rows += nested_cv(method)
nested = pd.DataFrame(nested_rows)

none fold 1 ROC-AUC 0.894


none fold 2 ROC-AUC 0.905


none fold 3 ROC-AUC 0.884


none fold 4 ROC-AUC 0.86


none fold 5 ROC-AUC 0.91


...

undersample fold 1 ROC-AUC 0.893


undersample fold 2 ROC-AUC 0.903


undersample fold 3 ROC-AUC 0.882


undersample fold 4 ROC-AUC 0.861


undersample fold 5 ROC-AUC 0.912


In [17]:
#the 5 test folds for each method
nested.round(3)

,method,fold,ROC-AUC,PR-AUC,test companies,bankrupt companies in test
0,none,1,0.894,0.082,1418,45
1,none,2,0.905,0.110,1428,40
2,none,3,0.884,0.086,1434,43
3,none,4,0.860,0.056,1430,38
4,none,5,0.910,0.131,1427,38
5,weights,1,0.897,0.081,1418,45
6,weights,2,0.904,0.106,1428,40
7,weights,3,0.879,0.090,1434,43
8,weights,4,0.863,0.049,1430,38
9,weights,5,0.910,0.094,1427,38


In [18]:
#what we report: the average over the 5 test folds, and how much it moves from fold to fold
nested_summary = nested.groupby("method")[["ROC-AUC", "PR-AUC"]].agg(["mean", "std"]).round(3)
nested_summary.loc[["none", "weights", "oversample", "undersample"]]

ROC-AUC        PR-AUC       
               mean    std   mean    std
method                                  
none          0.890  0.020  0.093  0.029
weights       0.890  0.019  0.084  0.021
oversample    0.889  0.016  0.084  0.024
undersample   0.890  0.020  0.098  0.042